# 🏥 RareNav Seoul — Full Pipeline
**One notebook. Runs everything.**

| Step | What happens |
|------|-------------|
| 0 | Install dependencies |
| 1 | Load Gemma 4 via Unsloth |
| 2 | Build rare disease corpus (OMIM, Orphanet, HPO, PubMed) |
| 3 | Embed + index into Qdrant |
| 4 | Build four-agent pipeline |
| 5 | SFT fine-tuning |
| 6 | GRPO reinforcement learning |
| 7 | Evaluation harness |
| 8 | Gradio demo app |

**Kaggle setup:** GPU T4 x2 · Internet ON · Accelerator GPU

Each step is independently runnable — skip steps you've already done by setting the `RUN_*` flags in Cell 1.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — CONFIG: toggle which steps to run
# ════════════════════════════════════════════════════════════
import os

# ── Step toggles ─────────────────────────────────────────────
RUN_INSTALL       = True   # Step 0: pip installs (skip after first run)
RUN_CORPUS        = True   # Step 2: build corpus from OMIM/Orphanet/HPO/PubMed
RUN_EMBED         = True   # Step 3: embed + index into Qdrant
RUN_SFT           = True   # Step 5: supervised fine-tuning
RUN_GRPO          = True   # Step 6: GRPO reinforcement learning
RUN_EVAL          = True   # Step 7: evaluation harness
RUN_DEMO          = True   # Step 8: launch Gradio demo

# ── Paths ────────────────────────────────────────────────────
DATA_DIR          = 'data'
CORPUS_PATH       = f'{DATA_DIR}/corpus_raw.jsonl'
QDRANT_PATH       = f'{DATA_DIR}/qdrant_store'
SFT_DATA_PATH     = f'{DATA_DIR}/sft_dataset.jsonl'
MODEL_OUTPUT_DIR  = 'models/rarenav-sft'
GRPO_OUTPUT_DIR   = 'models/rarenav-grpo'

# ── Model ────────────────────────────────────────────────────
BASE_MODEL        = 'unsloth/gemma-4-E4B-it'   # change to 26B-A4B-it for larger GPU
COLLECTION_NAME   = 'rarenav_corpus'
EMBED_MODEL       = 'snunlp/KR-SBERT-V40K-klueNLI-augSTS'

# ── Training ─────────────────────────────────────────────────
SFT_EPOCHS        = 2
SFT_MAX_STEPS     = 200    # set to -1 to use epochs instead
GRPO_MAX_STEPS    = 100    # enough for a meaningful reward curve
LORA_R            = 16
LORA_ALPHA        = 32

# ── Secrets (set in Kaggle → Add-ons → Secrets) ──────────────
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    OMIM_API_KEY  = _s.get_secret('OMIM_API_KEY')
    ENTREZ_EMAIL  = _s.get_secret('ENTREZ_EMAIL')
    HF_TOKEN      = _s.get_secret('HF_TOKEN')       # for pushing weights
    print('✓ Secrets loaded')
except Exception:
    OMIM_API_KEY  = ''
    ENTREZ_EMAIL  = 'rarenav@example.com'
    HF_TOKEN      = ''
    print('⚠ Secrets not found — OMIM skipped, HF push disabled')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Config ready.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — STEP 0: Install dependencies
# ════════════════════════════════════════════════════════════
if RUN_INSTALL:
    print('Installing dependencies...')
    os.system('pip install -q unsloth')
    os.system('pip install -q qdrant-client sentence-transformers')
    os.system('pip install -q biopython requests tqdm pandas gradio')
    os.system('wget -q https://github.com/obophenotype/human-phenotype-ontology/releases/latest/download/hp.obo -O data/hp.obo')
    print('✓ Done. Restart kernel if prompted by Unsloth.')
else:
    print('Skipping install.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — STEP 1: Load Gemma 4
# ════════════════════════════════════════════════════════════
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name    = BASE_MODEL,
    dtype         = None,
    max_seq_length= 8192,
    load_in_4bit  = True,
    full_finetuning = False,
    device_map    = 'balanced',
)
tokenizer = get_chat_template(tokenizer, chat_template='gemma-4')
print(f'✓ Model loaded: {BASE_MODEL}')
print(f'  Devices: {list(set(model.hf_device_map.values()))}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — STEP 2: Build corpus
# ════════════════════════════════════════════════════════════
import json, time, re, requests
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm
from Bio import Entrez

Entrez.email = ENTREZ_EMAIL

def append_doc(doc):
    with open(CORPUS_PATH, 'a', encoding='utf-8') as f:
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

# ── OMIM ─────────────────────────────────────────────────────
SEED_MIMS = [
    261600, 276600, 232200, 245200, 248600,   # metabolic
    230800, 230900, 231000, 200100, 301500,   # lysosomal
    100800, 166200, 154700, 227650,           # connective tissue / skeletal
    535000, 251880, 256000,                   # mitochondrial
    105400, 209850, 311200, 300624,           # neurodevelopmental
    102700, 202500, 300755,                   # immunodeficiency
]

def fetch_omim(mims):
    if not OMIM_API_KEY:
        print('  ⚠ No OMIM key — skipping'); return 0
    count = 0
    for i in range(0, len(mims), 20):
        chunk = mims[i:i+20]
        r = requests.get('https://api.omim.org/api/entry', params={
            'mimNumber': ','.join(str(m) for m in chunk),
            'include': 'clinicalSynopsis,text,geneMap',
            'format': 'json', 'apiKey': OMIM_API_KEY
        }, timeout=30)
        for entry in r.json()['omim']['entryList']:
            e = entry['entry']
            sections = {s['title']: s.get('textSectionContent','') for s in e.get('textSectionList',[])}
            append_doc({
                'id': f"omim_{e['mimNumber']}", 'source': 'omim',
                'title_en': e.get('titles',{}).get('preferredTitle',''),
                'text_en': '\n\n'.join(f"## {k}\n{v}" for k,v in sections.items() if v),
                'metadata': {'mim_number': e['mimNumber'],
                             'gene_symbols': [g.get('geneSymbol','') for g in e.get('geneMap',{}).get('geneMapList',[])]}
            })
            count += 1
        time.sleep(0.5)
    return count

# ── Orphanet ──────────────────────────────────────────────────
def parse_orphanet(xml_path='data/orphanet_product1.xml'):
    if not Path(xml_path).exists():
        print(f'  ⚠ {xml_path} not found — download from orphadata.com'); return 0
    count = 0
    for disorder in ET.parse(xml_path).getroot().iter('Disorder'):
        code = disorder.findtext('OrphaCode','')
        name = disorder.findtext('Name','')
        summary = disorder.findtext('SummaryInformation/TextSection/Contents','')
        synonyms = [s.text for s in disorder.findall('SynonymList/Synonym') if s.text]
        append_doc({
            'id': f'orphanet_{code}', 'source': 'orphanet',
            'title_en': name,
            'text_en': summary or f'Rare disease: {name}. Synonyms: {", ".join(synonyms)}.',
            'metadata': {'orpha_code': code, 'synonyms': synonyms}
        })
        count += 1
    return count

# ── HPO ───────────────────────────────────────────────────────
def parse_hpo(obo_path='data/hp.obo'):
    if not Path(obo_path).exists():
        print(f'  ⚠ {obo_path} not found'); return 0
    count, cur = 0, {}
    with open(obo_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line == '[Term]':
                if cur.get('id_raw','').startswith('HP:'):
                    append_doc({'id': 'hpo_'+cur['id_raw'].replace(':','_'), 'source': 'hpo',
                                'title_en': cur.get('name',''),
                                'text_en': cur.get('def','') + (' Synonyms: '+', '.join(cur.get('syns',[])) if cur.get('syns') else ''),
                                'metadata': {'hpo_id': cur['id_raw'], 'synonyms': cur.get('syns',[])}})
                    count += 1
                cur = {'syns': []}
            elif line.startswith('id: '):     cur['id_raw'] = line[4:]
            elif line.startswith('name: '):   cur['name']   = line[6:]
            elif line.startswith('def: '):    cur['def']    = line[5:].split('"')[1] if '"' in line else line[5:]
            elif line.startswith('synonym: '):
                p = line.split('"')
                if len(p) > 1: cur['syns'].append(p[1])
    return count

# ── PubMed ────────────────────────────────────────────────────
PUBMED_QUERIES = [
    'rare disease diagnosis Korea case report',
    'inborn error metabolism Korean patient',
    'lysosomal storage disease Korea',
    'orphan disease clinical features treatment review',
]

def fetch_pubmed(max_per_query=150):
    count, seen = 0, set()
    for q in PUBMED_QUERIES:
        handle = Entrez.esearch(db='pubmed', term=q, retmax=max_per_query, sort='relevance')
        ids = Entrez.read(handle)['IdList']; handle.close()
        for i in range(0, len(ids), 50):
            handle = Entrez.efetch(db='pubmed', id=','.join(ids[i:i+50]), rettype='abstract', retmode='xml')
            records = Entrez.read(handle); handle.close()
            for art in records.get('PubmedArticle', []):
                try:
                    ml = art['MedlineCitation']
                    pmid = str(ml['PMID'])
                    if pmid in seen: continue
                    seen.add(pmid)
                    a = ml['Article']
                    abstract = ' '.join(str(x) for x in a.get('Abstract',{}).get('AbstractText',[]))
                    append_doc({'id': f'pubmed_{pmid}', 'source': 'pubmed',
                                'title_en': str(a.get('ArticleTitle','')),
                                'text_en': abstract,
                                'metadata': {'pmid': pmid}})
                    count += 1
                except Exception: continue
            time.sleep(0.4)
    return count

# ── Run ───────────────────────────────────────────────────────
if RUN_CORPUS:
    if Path(CORPUS_PATH).exists(): Path(CORPUS_PATH).unlink()
    print('Building corpus...')
    n1 = fetch_omim(SEED_MIMS);     print(f'  OMIM: {n1}')
    n2 = parse_orphanet();           print(f'  Orphanet: {n2}')
    n3 = parse_hpo();                print(f'  HPO: {n3}')
    n4 = fetch_pubmed();             print(f'  PubMed: {n4}')
    total = sum(1 for _ in open(CORPUS_PATH))
    print(f'✓ Corpus: {total:,} documents → {CORPUS_PATH}')
else:
    total = sum(1 for _ in open(CORPUS_PATH)) if Path(CORPUS_PATH).exists() else 0
    print(f'Skipping corpus build. Existing docs: {total:,}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — STEP 3: Embed + index into Qdrant
# ════════════════════════════════════════════════════════════
import uuid
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBED_MODEL)
EMBED_DIM = 768
BATCH_SIZE = 64

def chunk_doc(doc, chunk_size=400, overlap=80):
    text  = doc.get('text_en','').strip()
    title = doc.get('title_en','').strip()
    if not text or len(text) < 30: return []
    full = f'{title}\n\n{text}' if title else text
    words = full.split()
    base  = {'doc_id': doc['id'], 'source': doc['source'], 'title': title, 'metadata': doc.get('metadata',{})}
    if len(words) <= chunk_size:
        return [{**base, 'chunk_id': str(uuid.uuid4()), 'text': full}]
    chunks = []
    for start in range(0, len(words), chunk_size - overlap):
        chunks.append({**base, 'chunk_id': str(uuid.uuid4()), 'text': ' '.join(words[start:start+chunk_size]),
                       'chunk_index': start // (chunk_size - overlap)})
    return chunks

def upsert_batch(client, chunks):
    vecs = embedder.encode([c['text'] for c in chunks], batch_size=BATCH_SIZE, show_progress_bar=False)
    client.upsert(collection_name=COLLECTION_NAME, points=[
        PointStruct(id=str(uuid.uuid5(uuid.NAMESPACE_DNS, c['chunk_id'])),
                    vector=vecs[i].tolist(),
                    payload={'doc_id': c['doc_id'], 'source': c['source'],
                             'text': c['text'], 'title': c['title'], 'metadata': c['metadata']})
        for i, c in enumerate(chunks)
    ])

if RUN_EMBED:
    qdrant = QdrantClient(path=QDRANT_PATH)
    existing = [c.name for c in qdrant.get_collections().collections]
    if COLLECTION_NAME not in existing:
        qdrant.create_collection(COLLECTION_NAME, vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE))
    batch, total_indexed = [], 0
    with open(CORPUS_PATH, encoding='utf-8') as f:
        for line in tqdm(f, desc='Indexing'):
            for chunk in chunk_doc(json.loads(line)):
                batch.append(chunk)
                if len(batch) >= BATCH_SIZE:
                    upsert_batch(qdrant, batch); total_indexed += len(batch); batch = []
    if batch: upsert_batch(qdrant, batch); total_indexed += len(batch)
    print(f'✓ Indexed {total_indexed:,} chunks')
else:
    qdrant = QdrantClient(path=QDRANT_PATH)
    print(f'Skipping embed. Collection: {qdrant.get_collection(COLLECTION_NAME).vectors_count:,} vectors')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — STEP 4: Language bridge + four-agent pipeline
# ════════════════════════════════════════════════════════════
import re, json
from dataclasses import dataclass, field, asdict
from typing import Literal

# ── Language bridge ───────────────────────────────────────────
_HANGUL = re.compile(r'[\uAC00-\uD7A3\u1100-\u11FF\u3130-\u318F]')

def detect_lang(text):
    nws = [c for c in text if not c.isspace()]
    return 'ko' if nws and len(_HANGUL.findall(text))/len(nws) > 0.10 else 'en'

def _generate(prompt, max_new_tokens=512, temperature=0.1):
    inputs = tokenizer.apply_chat_template(
        [{'role':'user','content':prompt}], tokenize=True,
        add_generation_prompt=True, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens,
                             temperature=temperature, do_sample=True,
                             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True).strip()

def translate_to_en(ko_text):
    return _generate(f'Translate this Korean medical text to English. Output translation only:\n{ko_text}')

def translate_to_ko(en_text):
    return _generate(f'다음 영어 의학 텍스트를 한국어로 번역하세요. 번역문만 출력하세요:\n{en_text}')

# ── Qdrant retriever ──────────────────────────────────────────
def retrieve(query, translated_query=None, top_k=8):
    q1 = embedder.encode(query).tolist()
    res = {r.payload['doc_id']: r for r in qdrant.search(COLLECTION_NAME, q1, limit=top_k)}
    if translated_query:
        q2 = embedder.encode(translated_query).tolist()
        for r in qdrant.search(COLLECTION_NAME, q2, limit=top_k):
            if r.payload['doc_id'] not in res or r.score > res[r.payload['doc_id']].score:
                res[r.payload['doc_id']] = r
    return sorted(res.values(), key=lambda r: r.score, reverse=True)[:top_k]

# ── Agent 1: Intake ───────────────────────────────────────────
def agent_intake(raw_note):
    print('[1/4] Intake agent...')
    lang = detect_lang(raw_note)
    en_note = translate_to_en(raw_note) if lang == 'ko' else raw_note

    prompt = """You are a clinical genetics expert. Extract HPO terms and patient demographics from this clinical note.
Respond ONLY with valid JSON — no markdown, no explanation:
{"patient_meta":{"age":"","sex":"","onset":""},"hpo_terms":[{"hpo_id":"HP:XXXXXXX","term":"","confidence":0.0,"evidence_text":""}]}

Clinical note:
""" + en_note

    raw = _generate(prompt, max_new_tokens=1024, temperature=0.05)
    # robust JSON extraction
    for pattern in [r'```json\s*(\{.*?\})\s*```', r'```\s*(\{.*?\})\s*```', r'(\{.*\})']:
        m = re.search(pattern, raw, re.DOTALL)
        if m:
            try: parsed = json.loads(m.group(1)); break
            except: pass
    else:
        parsed = {'patient_meta': {}, 'hpo_terms': []}

    hpo_terms = parsed.get('hpo_terms', [])
    hpo_terms.sort(key=lambda t: t.get('confidence', 0), reverse=True)
    print(f'  ✓ {len(hpo_terms)} HPO terms | lang={lang}')
    return {'en_note': en_note, 'raw_note': raw_note, 'lang': lang,
            'patient_meta': parsed.get('patient_meta', {}), 'hpo_terms': hpo_terms}

# ── Agent 2: Retrieval ────────────────────────────────────────
def agent_retrieval(intake):
    print('[2/4] Retrieval agent...')
    high_conf = [t for t in intake['hpo_terms'] if t.get('confidence', 0) >= 0.7]
    terms = high_conf[:6] or intake['hpo_terms'][:4]
    query = ' '.join(t['term'] for t in terms) + ' rare genetic disease'
    translated = query if intake['lang'] == 'ko' else None
    chunks = retrieve(intake['raw_note'], translated_query=translated, top_k=8)

    # Fallback: broaden if low confidence
    if not chunks or chunks[0].score < 0.35:
        print('  Low confidence — broadening query...')
        broad = ' '.join(t['term'] for t in intake['hpo_terms'][:3]) + ' inborn error metabolism'
        chunks = retrieve(broad, top_k=8)

    # Live PubMed
    pubmed_hits = []
    try:
        term_str = ' AND '.join(f'"{t["term"]}"' for t in high_conf[:3])
        handle = Entrez.esearch(db='pubmed', term=term_str + ' rare disease', retmax=3)
        ids = Entrez.read(handle)['IdList']; handle.close()
        if ids:
            handle = Entrez.efetch(db='pubmed', id=','.join(ids), rettype='abstract', retmode='xml')
            recs = Entrez.read(handle); handle.close()
            for art in recs.get('PubmedArticle', []):
                ml = art['MedlineCitation']
                pmid = str(ml['PMID'])
                a = ml['Article']
                pubmed_hits.append({'id': pmid, 'title': str(a.get('ArticleTitle','')),
                                    'snippet': ' '.join(str(x) for x in a.get('Abstract',{}).get('AbstractText',[]))[:300]})
    except Exception as e:
        print(f'  PubMed fetch error: {e}')

    print(f'  ✓ {len(chunks)} local chunks | {len(pubmed_hits)} PubMed hits')
    return {'chunks': chunks, 'pubmed_hits': pubmed_hits}

# ── Agent 3: Reasoning ────────────────────────────────────────
def agent_reasoning(intake, retrieval):
    print('[3/4] Reasoning agent...')
    context_parts = []
    for r in retrieval['chunks'][:6]:
        src = r.payload.get('source',''); title = r.payload.get('title','')
        meta = r.payload.get('metadata', {})
        cit = f"OMIM #{meta.get('mim_number','')}" if src=='omim' else \
              f"PMID {meta.get('pmid','')}" if src=='pubmed' else \
              f"Orphanet #{meta.get('orpha_code','')}" if src=='orphanet' else src.upper()
        context_parts.append(f'[{cit}] {title}\n{r.payload["text"][:400]}')
    for h in retrieval['pubmed_hits'][:3]:
        context_parts.append(f'[PubMed {h["id"]}] {h["title"]}\n{h["snippet"]}')
    context = '\n\n'.join(context_parts)

    hpo_str = '\n'.join(f'  - {t["term"]} ({t["hpo_id"]}) [{t.get("confidence",0):.0%}]'
                        for t in intake['hpo_terms'][:10])
    meta = intake['patient_meta']

    prompt = f"""You are a clinical genetics specialist. Generate a differential diagnosis for rare diseases.

PATIENT: {meta.get('age','')} {meta.get('sex','')} | Onset: {meta.get('onset','')}
HPO TERMS:\n{hpo_str}
LITERATURE:\n{context}
NOTE: {intake['en_note'][:500]}

Respond ONLY with JSON:
{{"reasoning_trace":"<step-by-step reasoning>","overall_confidence":0.0,
  "diagnoses":[{{"rank":1,"disease_name":"","omim_id":"","orpha_code":"",
    "confidence":0.0,"matching_features":[],"missing_features":[],
    "next_tests":[],"evidence_summary":"","citations":[]}}]}}"""

    raw = _generate(prompt, max_new_tokens=2048, temperature=0.1)
    for pattern in [r'```json\s*(\{.*?\})\s*```', r'```\s*(\{.*?\})\s*```', r'(\{.*\})']:
        m = re.search(pattern, raw, re.DOTALL)
        if m:
            try: parsed = json.loads(m.group(1)); break
            except: pass
    else:
        parsed = {'diagnoses': [], 'reasoning_trace': '', 'overall_confidence': 0.0}

    diagnoses = sorted(parsed.get('diagnoses', []), key=lambda d: d.get('confidence',0), reverse=True)
    for i, d in enumerate(diagnoses): d['rank'] = i+1
    print(f'  ✓ {len(diagnoses)} diagnoses | top: {diagnoses[0]["disease_name"] if diagnoses else "none"}')
    return {'diagnoses': diagnoses, 'reasoning': parsed.get('reasoning_trace',''),
            'confidence': parsed.get('overall_confidence', 0.0)}

# ── Agent 4: Report ───────────────────────────────────────────
def agent_report(intake, retrieval, reasoning):
    print('[4/4] Report agent...')
    diagnoses = reasoning['diagnoses']
    if not diagnoses:
        return {'text': 'No differential generated.', 'lang': intake['lang']}

    if intake['lang'] == 'ko':
        dx_str = '\n'.join(
            f"{d['rank']}. {d['disease_name']} (신뢰도: {d.get('confidence',0):.0%})\n"
            f"   근거: {d.get('evidence_summary','')}\n"
            f"   권장 검사: {', '.join(d.get('next_tests',[])[:3])}"
            for d in diagnoses[:3])
        hpo_str = ', '.join(f"{t['term']} ({t['hpo_id']})" for t in intake['hpo_terms'][:5])
        meta = intake['patient_meta']
        prompt = f"""임상 유전학 전문의로서 다음 희귀질환 감별 진단 결과로 한국어 의뢰서를 작성하세요.

[환자] {meta.get('age','')} {meta.get('sex','')} | 발병: {meta.get('onset','')}
[주요 표현형] {hpo_str}
[감별 진단]\n{dx_str}

형식: 1)환자요약 2)주요소견 3)감별진단(근거포함) 4)권장검사 5)의뢰사유 6)참고문헌
의학용어는 한국어+영어 병기. 전문적이고 간결하게."""
        report_text = _generate(prompt, max_new_tokens=1024, temperature=0.2)
    else:
        sep = '─' * 58
        meta = intake['patient_meta']
        lines = ['═'*58, 'RARENAV SEOUL — CLINICAL REFERRAL BRIEF', '═'*58, '',
                 f"Patient : {meta.get('age','')} {meta.get('sex','')}",
                 f"Onset   : {meta.get('onset','')}", '']
        lines += ['KEY PHENOTYPIC FEATURES', sep]
        for t in intake['hpo_terms'][:6]:
            lines.append(f"  • {t['term']} ({t['hpo_id']}) [{t.get('confidence',0):.0%}]")
        lines += ['', 'DIFFERENTIAL DIAGNOSIS', sep]
        for dx in diagnoses[:3]:
            ids = ' | '.join(filter(None, [dx.get('omim_id',''), dx.get('orpha_code','')]))
            lines += [f"#{dx['rank']} {dx['disease_name']}" + (f' [{ids}]' if ids else ''),
                      f"   Confidence : {dx.get('confidence',0):.0%}",
                      f"   Evidence   : {dx.get('evidence_summary','')}",
                      f"   Matching   : {', '.join(dx.get('matching_features',[])[:4])}",
                      f"   Next tests : {', '.join(dx.get('next_tests',[])[:3])}", '']
        lines += ['═'*58, 'RareNav Seoul | Gemma 4 + Unsloth',
                  '⚠ For specialist review only. Not a final diagnosis.', '═'*58]
        report_text = '\n'.join(lines)

    print(f'  ✓ Report generated ({len(report_text)} chars, lang={intake["lang"]})')
    return {'text': report_text, 'lang': intake['lang'], 'diagnoses': diagnoses}

# ── Pipeline orchestrator ─────────────────────────────────────
import time as _time

def run_pipeline(raw_query):
    t0 = _time.time()
    timings = {}
    print(f'\n{"═"*58}'); print(f'  Query: {raw_query[:60]}...'); print('═'*58)
    t = _time.time(); intake    = agent_intake(raw_query);          timings['intake']    = _time.time()-t
    t = _time.time(); retrieval = agent_retrieval(intake);          timings['retrieval'] = _time.time()-t
    t = _time.time(); reasoning = agent_reasoning(intake,retrieval);timings['reasoning'] = _time.time()-t
    t = _time.time(); report    = agent_report(intake,retrieval,reasoning); timings['report'] = _time.time()-t
    total = _time.time() - t0
    print(f'\nTimings: ' + ' | '.join(f'{k}={v:.1f}s' for k,v in timings.items()) + f' | total={total:.1f}s')
    return {'intake': intake, 'retrieval': retrieval, 'reasoning': reasoning, 'report': report, 'timings': timings}

print('✓ Pipeline defined.')

# Quick smoke test
test = run_pipeline('발달 지연, 경련, 소변에서 쥐 냄새가 나는 6세 남아. 혈중 페닐알라닌 25 mg/dL.')
print('\n' + '='*58)
print(test['report']['text'])

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — STEP 5: Build SFT dataset + fine-tune
# ════════════════════════════════════════════════════════════
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import pandas as pd

# ── SFT training pairs ────────────────────────────────────────
# Each example: clinical note → correct structured differential
# Ground truth from OMIM case studies + known phenotype-disease pairs
SFT_EXAMPLES = [
    {'note': 'Newborn male with hypotonia, poor feeding, and elevated blood phenylalanine (18 mg/dL). Urine positive for phenylketones. Parents are consanguineous.',
     'differential': '{"diagnoses":[{"rank":1,"disease_name":"Phenylketonuria (PKU)","omim_id":"OMIM:261600","confidence":0.95,"matching_features":["Hyperphenylalaninemia","Phenylalanuria","Neonatal hypotonia"],"missing_features":["Intellectual disability (would develop without treatment)"],"next_tests":["PAH gene sequencing","BH4 loading test","Newborn screening confirmation"],"evidence_summary":"Classic PKU presentation with elevated phenylalanine and urine phenylketones in consanguineous family."}]}'},
    {'note': '8-month-old female with progressive hypotonia, hypertrophic cardiomyopathy on echo, and CK 1200 U/L. Acid alpha-glucosidase (GAA) activity 1% of normal.',
     'differential': '{"diagnoses":[{"rank":1,"disease_name":"Pompe Disease (Glycogen Storage Disease Type II)","omim_id":"OMIM:232300","confidence":0.97,"matching_features":["Hypotonia","Hypertrophic cardiomyopathy","Elevated CK","GAA deficiency"],"missing_features":["Respiratory insufficiency"],"next_tests":["GAA enzyme activity confirmation","GAA gene sequencing","Muscle biopsy","Echocardiogram follow-up"],"evidence_summary":"Infantile-onset Pompe confirmed by near-absent GAA activity with classic cardiopulmonary presentation."}]}'},
    {'note': '12-year-old female with progressive ataxia, vertical supranuclear gaze palsy, hepatosplenomegaly, and foam cells on bone marrow biopsy. Elevated oxysterols.',
     'differential': '{"diagnoses":[{"rank":1,"disease_name":"Niemann-Pick Disease Type C","omim_id":"OMIM:257220","orpha_code":"ORPHA:646","confidence":0.91,"matching_features":["Ataxia","Vertical supranuclear gaze palsy","Hepatosplenomegaly","Foam cells on biopsy","Elevated oxysterols"],"missing_features":["Cataplexy","Dystonia"],"next_tests":["NPC1/NPC2 gene sequencing","Filipin staining","Plasma oxysterol panel"],"evidence_summary":"Pathognomonic triad of VSGP, ataxia, and organomegaly with foam cells strongly indicates NPC."}]}'},
    {'note': '3-year-old male with coarse facial features, corneal clouding, hepatosplenomegaly, skeletal dysplasia, and developmental regression. Urine heparan/dermatan sulfate elevated.',
     'differential': '{"diagnoses":[{"rank":1,"disease_name":"Mucopolysaccharidosis Type I (Hurler Syndrome)","omim_id":"OMIM:607014","orpha_code":"ORPHA:93473","confidence":0.93,"matching_features":["Coarse facial features","Corneal clouding","Hepatosplenomegaly","Dysostosis multiplex","MPS in urine"],"missing_features":["Hearing loss"],"next_tests":["Alpha-L-iduronidase enzyme assay","IDUA gene sequencing","Urine GAG quantification"],"evidence_summary":"Classic MPS-I with multi-system involvement and elevated urinary GAGs."}]}'},
    {'note': 'Young adult female with haemolytic anaemia, splenomegaly, and bone marrow showing lipid-laden macrophages (Gaucher cells). Beta-glucocerebrosidase activity markedly reduced.',
     'differential': '{"diagnoses":[{"rank":1,"disease_name":"Gaucher Disease Type 1","omim_id":"OMIM:230800","orpha_code":"ORPHA:355","confidence":0.96,"matching_features":["Haemolytic anaemia","Splenomegaly","Gaucher cells on marrow","GBA deficiency"],"missing_features":["Bone pain/crisis","Thrombocytopenia"],"next_tests":["GBA enzyme activity","GBA gene sequencing","Chitotriosidase level"],"evidence_summary":"Non-neuronopathic Gaucher confirmed by Gaucher cells and absent GBA activity."}]}'},
]

def format_sft_example(example):
    user = f"""You are a clinical genetics specialist. Given this clinical note, output a structured differential diagnosis as JSON only.

Clinical note: {example['note']}"""
    return tokenizer.apply_chat_template(
        [{'role':'user','content':user}, {'role':'assistant','content':example['differential']}],
        tokenize=False, add_generation_prompt=False
    )

if RUN_SFT:
    print('Preparing SFT dataset...')

    # Format + save dataset
    formatted = [{'text': format_sft_example(ex)} for ex in SFT_EXAMPLES]
    # Augment by running pipeline on HPO-disease pairs from corpus
    # (in practice: add 100–500 pairs from OMIM case reports)
    sft_dataset = Dataset.from_list(formatted)
    print(f'  SFT dataset: {len(sft_dataset)} examples')

    # Add LoRA adapters
    model = FastModel.get_peft_model(
        model,
        r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_dropout=0.0, bias='none', use_gradient_checkpointing='unsloth',
    )

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=sft_dataset,
        args=SFTConfig(
            output_dir=MODEL_OUTPUT_DIR,
            num_train_epochs=SFT_EPOCHS,
            max_steps=SFT_MAX_STEPS,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            warmup_steps=10,
            learning_rate=2e-4,
            logging_steps=10,
            save_steps=50,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
            optim='adamw_8bit',
            max_seq_length=4096,
            dataset_text_field='text',
        )
    )
    print('Starting SFT training...')
    trainer.train()
    model.save_pretrained(MODEL_OUTPUT_DIR)
    tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
    print(f'✓ SFT complete → {MODEL_OUTPUT_DIR}')
else:
    print('Skipping SFT.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — STEP 6: GRPO reinforcement learning
# ════════════════════════════════════════════════════════════
from trl import GRPOTrainer, GRPOConfig

# ── Reward functions ──────────────────────────────────────────
def reward_disease_in_differential(completions, ground_truth_disease, **kwargs):
    """Reward: correct disease appears in top-3 of generated differential."""
    rewards = []
    for completion in completions:
        try:
            text = completion[0]['content'] if isinstance(completion, list) else completion
            m = re.search(r'(\{.*\})', text, re.DOTALL)
            parsed = json.loads(m.group(1)) if m else {}
            top3 = [d.get('disease_name','').lower() for d in parsed.get('diagnoses',[])[:3]]
            gt = ground_truth_disease.lower()
            # Partial match reward: exact=1.0, partial=0.5, miss=0.0
            if any(gt in d or d in gt for d in top3): rewards.append(1.0)
            elif any(gt.split()[0] in d for d in top3): rewards.append(0.5)
            else: rewards.append(0.0)
        except: rewards.append(0.0)
    return rewards

def reward_citation_coverage(completions, **kwargs):
    """Reward: each diagnosis has at least one citation."""
    rewards = []
    for completion in completions:
        try:
            text = completion[0]['content'] if isinstance(completion, list) else completion
            m = re.search(r'(\{.*\})', text, re.DOTALL)
            parsed = json.loads(m.group(1)) if m else {}
            diagnoses = parsed.get('diagnoses', [])
            if not diagnoses: rewards.append(0.0); continue
            cited = sum(1 for d in diagnoses if d.get('citations'))
            rewards.append(cited / len(diagnoses))
        except: rewards.append(0.0)
    return rewards

def reward_no_hallucination(completions, **kwargs):
    """Reward: output is valid JSON with required fields (proxy for groundedness)."""
    rewards = []
    required_fields = {'disease_name', 'confidence', 'matching_features', 'next_tests'}
    for completion in completions:
        try:
            text = completion[0]['content'] if isinstance(completion, list) else completion
            m = re.search(r'(\{.*\})', text, re.DOTALL)
            parsed = json.loads(m.group(1)) if m else {}
            diagnoses = parsed.get('diagnoses', [])
            if not diagnoses: rewards.append(0.0); continue
            valid = sum(1 for d in diagnoses if required_fields.issubset(d.keys()))
            rewards.append(valid / len(diagnoses))
        except: rewards.append(0.0)
    return rewards

def combined_reward(completions, ground_truth_disease='', **kwargs):
    """Weighted combination of all reward signals."""
    r1 = reward_disease_in_differential(completions, ground_truth_disease, **kwargs)
    r2 = reward_citation_coverage(completions, **kwargs)
    r3 = reward_no_hallucination(completions, **kwargs)
    return [0.5*a + 0.3*b + 0.2*c for a, b, c in zip(r1, r2, r3)]

# ── GRPO dataset ──────────────────────────────────────────────
GRPO_EXAMPLES = [
    {'prompt': [{'role':'user','content': f'Clinical note: {ex["note"]}\nGenerate differential diagnosis JSON:'}],
     'ground_truth_disease': json.loads(ex['differential'])['diagnoses'][0]['disease_name']}
    for ex in SFT_EXAMPLES
]
grpo_dataset = Dataset.from_list(GRPO_EXAMPLES)

if RUN_GRPO:
    print('Starting GRPO training...')
    grpo_trainer = GRPOTrainer(
        model=model,
        processing_class=tokenizer,
        reward_funcs=[combined_reward],
        args=GRPOConfig(
            output_dir=GRPO_OUTPUT_DIR,
            max_steps=GRPO_MAX_STEPS,
            per_device_train_batch_size=1,
            gradient_accumulation_steps=4,
            num_generations=4,          # GRPO group size
            max_new_tokens=1024,
            learning_rate=5e-6,         # lower LR for RL stability
            kl_coeff=0.1,
            logging_steps=10,
            save_steps=50,
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),
        ),
        train_dataset=grpo_dataset,
    )
    grpo_trainer.train()
    model.save_pretrained(GRPO_OUTPUT_DIR)
    tokenizer.save_pretrained(GRPO_OUTPUT_DIR)
    print(f'✓ GRPO complete → {GRPO_OUTPUT_DIR}')

    # Push to HuggingFace
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        model.push_to_hub('rarenav-gemma4-grpo')
        tokenizer.push_to_hub('rarenav-gemma4-grpo')
        print('✓ Weights pushed to HuggingFace')
else:
    print('Skipping GRPO.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — STEP 7: Evaluation harness
# ════════════════════════════════════════════════════════════
import pandas as pd
from IPython.display import display, Markdown

EVAL_CASES = [
    {'note': 'Newborn with elevated phenylalanine 20 mg/dL, urine phenylketones positive, musty odor.',
     'gold': 'Phenylketonuria', 'gold_omim': 'OMIM:261600'},
    {'note': '8-month female, hypotonia, hypertrophic cardiomyopathy, GAA activity 1% of normal, CK 1200.',
     'gold': 'Pompe Disease', 'gold_omim': 'OMIM:232300'},
    {'note': '12-year-old, vertical supranuclear gaze palsy, ataxia, hepatosplenomegaly, foam cells, elevated oxysterols.',
     'gold': 'Niemann-Pick Disease Type C', 'gold_omim': 'OMIM:257220'},
    {'note': 'Toddler with coarse features, corneal clouding, organomegaly, urine heparan sulfate elevated.',
     'gold': 'Mucopolysaccharidosis Type I', 'gold_omim': 'OMIM:607014'},
    {'note': 'Young adult with haemolytic anaemia, massive splenomegaly, Gaucher cells on marrow, low GBA.',
     'gold': 'Gaucher Disease', 'gold_omim': 'OMIM:230800'},
    # Korean cases
    {'note': '발달 지연, 경련, 소변에서 쥐 냄새, 혈중 페닐알라닌 상승된 6세 남아.',
     'gold': 'Phenylketonuria', 'gold_omim': 'OMIM:261600'},
    {'note': '근육 약화, 심근비대, GAA 효소 활성 1% 미만인 영아.',
     'gold': 'Pompe Disease', 'gold_omim': 'OMIM:232300'},
]

def evaluate_case(case):
    result = run_pipeline(case['note'])
    diagnoses = result['reasoning'].get('diagnoses', [])
    top3_names = [d['disease_name'].lower() for d in diagnoses[:3]]
    gold = case['gold'].lower()

    top1_hit = any(gold in n or n in gold or gold.split()[0] in n for n in top3_names[:1])
    top3_hit = any(gold in n or n in gold or gold.split()[0] in n for n in top3_names)
    cited = sum(1 for d in diagnoses if d.get('citations'))
    has_tests = all(d.get('next_tests') for d in diagnoses[:3])

    # Hallucination proxy: valid JSON with all required fields
    required = {'disease_name','confidence','matching_features','next_tests','evidence_summary'}
    halluc_rate = 1 - (sum(1 for d in diagnoses if required.issubset(d.keys())) / max(len(diagnoses),1))

    return {
        'Case': case['gold'],
        'Lang': result['intake']['lang'].upper(),
        'Top-1 ✓': '✅' if top1_hit else '❌',
        'Top-3 ✓': '✅' if top3_hit else '❌',
        'N diagnoses': len(diagnoses),
        'Cited (%)': f"{cited/max(len(diagnoses),1):.0%}",
        'Tests ✓': '✅' if has_tests else '❌',
        'Halluc rate': f"{halluc_rate:.0%}",
        'Total (s)': round(sum(result['timings'].values()), 1),
    }

if RUN_EVAL:
    print('Running evaluation harness...')
    rows = []
    for i, case in enumerate(EVAL_CASES):
        print(f'  Case {i+1}/{len(EVAL_CASES)}: {case["gold"]} ({detect_lang(case["note"]).upper()})')
        rows.append(evaluate_case(case))

    df = pd.DataFrame(rows)
    display(Markdown('## Evaluation Results'))
    display(df)

    top1_acc = sum(1 for r in rows if r['Top-1 ✓'] == '✅') / len(rows)
    top3_acc = sum(1 for r in rows if r['Top-3 ✓'] == '✅') / len(rows)
    avg_halluc = sum(float(r['Halluc rate'].strip('%'))/100 for r in rows) / len(rows)
    print(f'\nSummary — Top-1: {top1_acc:.0%} | Top-3: {top3_acc:.0%} | Avg hallucination: {avg_halluc:.0%}')
else:
    print('Skipping eval.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — STEP 8: Gradio demo app
# ════════════════════════════════════════════════════════════
import gradio as gr
from IPython.display import display, Markdown

EXAMPLE_NOTES = [
    '환자: 6세 남아. 발달 지연, 반복 경련, 소변에서 쥐 냄새. 혈중 페닐알라닌 25 mg/dL.',
    '8-month-old female with progressive hypotonia, hypertrophic cardiomyopathy, CK 1200 U/L, GAA activity 1% of normal.',
    '12세 여아. 보행 장애, 수직 안구 운동 장애, 경미한 간비장 비대, 골수에서 거품세포 발견.',
]

def gradio_pipeline(note):
    if not note.strip():
        return 'Please enter a clinical note.', '', ''
    result = run_pipeline(note)

    # Agent trace
    intake    = result['intake']
    retrieval = result['retrieval']
    reasoning = result['reasoning']
    timings   = result['timings']

    trace = f"""**Language detected:** {intake['lang'].upper()}
**Patient:** {intake['patient_meta'].get('age','')} {intake['patient_meta'].get('sex','')} | Onset: {intake['patient_meta'].get('onset','')}
**HPO terms extracted:** {len(intake['hpo_terms'])}
{chr(10).join(f"  • {t['term']} ({t['hpo_id']}) [{t.get('confidence',0):.0%}]" for t in intake['hpo_terms'][:6])}

**Retrieved:** {len(retrieval['chunks'])} local docs + {len(retrieval['pubmed_hits'])} PubMed
**Top sources:**
{chr(10).join(f"  [{r.score:.3f}] ({r.payload['source']}) {r.payload['title'][:50]}" for r in retrieval['chunks'][:4])}

**Timing:** { ' | '.join(f"{k}={v:.1f}s" for k,v in timings.items()) }"""

    # Diagnoses table
    table_rows = []
    for dx in reasoning.get('diagnoses', [])[:5]:
        ids = ' / '.join(filter(None, [dx.get('omim_id',''), dx.get('orpha_code','')]))
        table_rows.append(f"| {dx['rank']} | {dx['disease_name']} | {dx.get('confidence',0):.0%} | {ids or '—'} | {', '.join(dx.get('next_tests',[])[:2])} |")
    table = '| Rank | Disease | Confidence | ID | Next Tests |\n|------|---------|------------|-----|------------|\n' + '\n'.join(table_rows)

    return result['report']['text'], trace, table

if RUN_DEMO:
    demo = gr.Interface(
        fn=gradio_pipeline,
        inputs=gr.Textbox(
            label='Clinical Note (Korean or English)',
            placeholder='Enter patient symptoms, lab values, clinical findings...',
            lines=6,
        ),
        outputs=[
            gr.Textbox(label='Referral Report', lines=25),
            gr.Markdown(label='Agent Trace'),
            gr.Markdown(label='Differential Diagnosis Table'),
        ],
        title='🏥 RareNav Seoul',
        description='Rare disease differential diagnosis · Powered by Gemma 4 + Unsloth · Accepts Korean or English',
        examples=[[n] for n in EXAMPLE_NOTES],
        theme=gr.themes.Soft(),
    )
    demo.launch(share=True)   # share=True gives a public URL on Kaggle
else:
    print('Skipping demo.')